# Verifier re-check (amendment 0010): the pinned Phi-4 on the smoke run's drafts

**Execution wrapper only.** Amendment 0010 replaced Phi-4-mini, which rejected all
40 drafts of the 2026-10-03 score smoke run with reasons false on their face, by
Phi-4 (14B), and allows one more attempt decided on **the same drafts**.
`tools/recheck_verifier.py` sends the verifier prompts stored in that run's ledger,
unchanged, to the model `configs/generators/b4_vte.yaml` pins, and writes
`review.csv` for the author to mark (amendment 0010(B)).

Accelerator **GPU T4** (one is enough: 9 GB), Internet **on**, input dataset
`slimihamda/faithfulids-tier-a-ledgers` (holds `recheck/phi4mini_smoke2/ledger.jsonl`).
No CICIDS data, no detector, no generator: about 20-30 minutes.


In [ ]:
# Launcher parameters ONLY.
import os
PINNED_REF = 'main'   # the terminal push replaces this with the exact commit
OUT = '/kaggle/working/recheck'
os.makedirs(OUT, exist_ok=True)
os.environ.update({'FAITHFULIDS_REF': PINNED_REF, 'RECHECK_OUT': OUT})
print('ref:', PINNED_REF)


In [ ]:
%%bash
set -e
set -o pipefail
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || { echo "NO GPU: Settings -> Accelerator -> GPU T4."; exit 1; }
rm -rf /tmp/faithful-ids
git clone --quiet https://github.com/SLIMIHamda/faithful-ids.git /tmp/faithful-ids
cd /tmp/faithful-ids && git checkout --quiet "$FAITHFULIDS_REF"
git rev-parse HEAD | tee "$RECHECK_OUT/PINNED_COMMIT.txt"
# Kaggle runs Python 3.13; --no-deps skips the pinned stack this tool does not need.
pip install -q -e . --no-deps --ignore-requires-python 2>&1 | tail -n 2
pip install -q pyyaml jsonschema 2>&1 | tail -n 2
python -c "import faithfulids, sys; print('faithfulids', faithfulids.__version__, '| python', sys.version.split()[0])"


In [ ]:
# Serve the pinned verifier under Ollama; every manifest blob must match the pins.
import glob, json, subprocess, time, urllib.request, yaml

mm = yaml.safe_load(open('/tmp/faithful-ids/configs/generators/b4_vte.yaml'))['verifier']['model']
subprocess.run('apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y zstd >/dev/null 2>&1', shell=True)
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
MODELS_DIR = '/tmp/ollama-models'
env = dict(os.environ, OLLAMA_HOST='127.0.0.1:11434', OLLAMA_KEEP_ALIVE='-1', OLLAMA_NUM_PARALLEL='1',
           OLLAMA_MAX_LOADED_MODELS='1', OLLAMA_MODELS=MODELS_DIR)
serve = subprocess.Popen(['ollama', 'serve'], env=env, stderr=subprocess.STDOUT,
                         stdout=open(f'{OUT}/ollama_serve.log', 'w'))
for _ in range(60):
    try:
        ver = json.load(urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=2))['version']
        break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit('ollama serve did not start: see recheck/ollama_serve.log')
need = tuple(int(x) for x in mm['ollama']['min_version'].split('.'))
assert tuple(int(x) for x in ver.split('-')[0].split('.')[:3]) >= need, f'Ollama {ver} is too old'
open(f'{OUT}/OLLAMA_VERSION.txt', 'w').write(ver + '\n')

t0 = time.time()
r = subprocess.run(['ollama', 'pull', mm['weights']['ref']], capture_output=True, text=True, env=env)
assert r.returncode == 0, f"ollama pull {mm['weights']['ref']} failed: {r.stderr[-2000:]}"
name, tag = mm['weights']['ref'].split(':')
manifest = json.load(open(f'{MODELS_DIR}/manifests/registry.ollama.ai/library/{name}/{tag}'))
got = {l['mediaType'].rsplit('.', 1)[-1]: l['digest'].split(':', 1)[1] for l in manifest['layers']}
want = {'model': mm['weights']['sha256'], 'template': mm['ollama']['template_sha256']}
if mm['ollama'].get('params_sha256'):
    want['params'] = mm['ollama']['params_sha256']
assert all(got.get(k) == v for k, v in want.items()), f'{got} is not the pinned {want}'
assert set(got) - {'license'} == set(want), f'unpinned layers {set(got) - set(want)}'
print(f"{mm['weights']['ref']} ok ({(time.time() - t0) / 60:.1f} min): {sorted(got)} match the pins | Ollama {ver}")

old = sorted(glob.glob('/kaggle/input/**/recheck/phi4mini_smoke2/ledger.jsonl', recursive=True))
assert old, 'attach slimihamda/faithfulids-tier-a-ledgers (it holds recheck/phi4mini_smoke2/ledger.jsonl)'
os.environ['RECHECK_OLD'] = os.path.dirname(old[0])
print('old verifier ledger:', old[0], '|', sum(1 for _ in open(old[0], encoding='utf-8')), 'calls')


In [ ]:
%%bash
set -o pipefail
cd /tmp/faithful-ids
python tools/recheck_verifier.py --old-ledger "$RECHECK_OLD" --out "$RECHECK_OUT/phi4_14b" 2>&1 \
  | tee "$RECHECK_OUT/recheck.log"
rc=$?
nvidia-smi > "$RECHECK_OUT/nvidia-smi.txt" 2>&1
cd /kaggle/working && rm -f recheck_artifacts.zip && zip -qr recheck_artifacts.zip recheck
ls -la recheck_artifacts.zip
exit $rc
